In [2]:
import polars as pl
import pandas as pd



In [3]:
tracking_df = pd.read_hdf("tracking_results/DeepEcoHAB_20260501_211615_007DLC_DekrW18_the_future_of_EcohabNov22shuffle1_snapshot_best-260_el.h5")

long_pdf = (
    tracking_df
    .stack(["scorer", "individuals", "bodyparts"])
    #.rename(["x","y"])
    .reset_index()
)
tracking_lf = pl.from_pandas(long_pdf).lazy().rename({'level_0':"frame"}).drop('scorer') #ograć rename wczytywaniem z hdf

/tmp/ipykernel_179866/1046192953.py:5: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(["scorer", "individuals", "bodyparts"])


In [4]:
tracking_lf.head().collect()

frame,individuals,bodyparts,x,y,likelihood
i64,str,str,f64,f64,f64
0,"""ind1""","""Dorsal1""",780.426636,282.12326,0.383084
0,"""ind1""","""Dorsal2""",780.251953,292.798309,0.383084
0,"""ind1""","""Dorsal3""",782.416443,306.688049,0.383084
0,"""ind1""","""Dorsal4""",782.880493,318.88559,0.383084
0,"""ind1""","""Ear_left""",776.271545,279.346252,0.383084


In [16]:

tracking_lf = tracking_lf.filter(
    pl.col('bodyparts') == "Dorsal2",#to decide, maybe centroid instead
).drop(pl.col("likelihood"))#.collect()

In [6]:
antenna_df = pl.scan_csv(
    "tracking_results/DeepEcoHAB_20260501_211615_antenna_data.csv", 
    has_header=False,
    separator=";",
    new_columns=['antenna', 'tag', 'timestamp', 'datetime'],
    infer_schema_length=1000
    ).with_columns(
        pl.col('timestamp').str.to_integer(base=16)
    )

video_index = 7 #to extract from filename suffix
frames_per_video = 108000

antenna_df = antenna_df.with_columns(
    (pl.col('antenna')==10).cum_sum().alias('frame')
).filter(
    pl.col('frame').is_between(frames_per_video*video_index, frames_per_video*(video_index+1)-1)
).with_columns(
    (pl.col('frame')-pl.col('frame').first()).alias('frame')
)

In [7]:
antenna_df.collect().to_pandas()

,antenna,tag,timestamp,datetime,frame
0,10,01000000,13881021,2026-05-02T04:16:25.187506,0
1,10,01000000,13881025,2026-05-02T04:16:25.234955,1
2,10,01000000,13881028,2026-05-02T04:16:25.267095,2
3,10,01000000,13881031,2026-05-02T04:16:25.299101,3
4,10,01000000,13881035,2026-05-02T04:16:25.330489,4
...,...,...,...,...,...
115307,10,01000000,14241014,2026-05-02T05:16:25.269040,107995
115308,10,01000000,14241017,2026-05-02T05:16:25.300795,107996
115309,10,01000000,14241021,2026-05-02T05:16:25.341808,107997
115310,10,01000000,14241024,2026-05-02T05:16:25.371307,107998


In [37]:
antenna_df.filter(pl.col('tag')=='02000000').collect().to_pandas()

,antenna,tag,timestamp,datetime,frame
0,1,02000000,14053249,2026-05-02T04:45:07.549451,51667


In [8]:
animal_detections = antenna_df.filter(
    pl.col("antenna") != 10#remove magic string
)

In [ ]:
animal_detections.collect().to_pandas()

,antenna,tag,timestamp,datetime,frame
0,5,F02AE61A,13883423,2026-05-02T04:16:49.228317,721
1,5,F02AE61A,13883427,2026-05-02T04:16:49.269069,722
2,5,F02AE61A,13883431,2026-05-02T04:16:49.300830,723
3,5,F02AE61A,13883435,2026-05-02T04:16:49.348346,724
4,5,F02AE61A,13883439,2026-05-02T04:16:49.380227,725
...,...,...,...,...,...
7307,1,F02AE61A,14235857,2026-05-02T05:15:33.706488,106448
7308,1,F02AE61A,14235861,2026-05-02T05:15:33.737894,106449
7309,1,F02AE61A,14235865,2026-05-02T05:15:33.785172,106450
7310,1,F02AE61A,14235869,2026-05-02T05:15:33.826882,106451


In [15]:
animal_detections = animal_detections.with_columns(
    (
        pl.struct(["antenna", "tag"]).ne(pl.struct(["antenna", "tag"]).shift())
        | pl.col("frame").diff().gt(5)
    )
    .cum_sum()
    .cast(pl.Int16)
    .alias("run_id")
    .fill_null(0)
)
animal_detections.collect().to_pandas()
#group by run id of antenna and tag, with columns under antenna - in the future take the actual timestamp instead of frame

,antenna,tag,timestamp,datetime,frame,run_id,jump
0,5,F02AE61A,13883423,2026-05-02T04:16:49.228317,721,0,NaN
1,5,F02AE61A,13883427,2026-05-02T04:16:49.269069,722,0,0.0
2,5,F02AE61A,13883431,2026-05-02T04:16:49.300830,723,0,0.0
3,5,F02AE61A,13883435,2026-05-02T04:16:49.348346,724,0,0.0
4,5,F02AE61A,13883439,2026-05-02T04:16:49.380227,725,0,0.0
...,...,...,...,...,...,...,...
7307,1,F02AE61A,14235857,2026-05-02T05:15:33.706488,106448,1459,948.0
7308,1,F02AE61A,14235861,2026-05-02T05:15:33.737894,106449,1459,948.0
7309,1,F02AE61A,14235865,2026-05-02T05:15:33.785172,106450,1459,948.0
7310,1,F02AE61A,14235869,2026-05-02T05:15:33.826882,106451,1459,948.0


In [22]:
antenna_with_coords = animal_detections.join(
    tracking_lf,
    on = 'frame',
    how = 'left'
)

In [24]:
antenna_with_coords.collect()

antenna,tag,timestamp,datetime,frame,run_id,jump,individuals,bodyparts,x,y
i64,str,i64,str,u32,i16,u32,str,str,f64,f64
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind1""","""Dorsal2""",1044.807983,744.289185
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind2""","""Dorsal2""",685.252319,483.831543
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind3""","""Dorsal2""",708.658997,480.565552
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind4""","""Dorsal2""",711.786011,360.191193
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind5""","""Dorsal2""",201.81424,310.670593
…,…,…,…,…,…,…,…,…,…,…
1,"""F02AE61A""",14235873,"""2026-05-02T05:15:33.870102""",106453,1459,948,"""ind4""","""Dorsal2""",711.827576,360.731415
1,"""F02AE61A""",14235873,"""2026-05-02T05:15:33.870102""",106453,1459,948,"""ind5""","""Dorsal2""",1131.467163,946.468811
1,"""F02AE61A""",14235873,"""2026-05-02T05:15:33.870102""",106453,1459,948,"""ind6""","""Dorsal2""",389.522766,170.759018


In [ ]:
antenna_with_coords = antenna_with_coords.filter((pl.col('tag') != '02000000') & (pl.col('tag') != '02000060')) #maybe use sanitation for that

antenna,tag,timestamp,datetime,frame,run_id,jump,individuals,bodyparts,x,y
i64,str,i64,str,u32,i16,u32,str,str,f64,f64
5,"""12F5FF19""",13885333,"""2026-05-02T04:17:08.327221""",1293,2,2,"""ind3""","""Dorsal2""",709.228699,480.140289
7,"""DD43E61A""",14104219,"""2026-05-02T04:53:37.265555""",66957,849,531,"""ind3""","""Dorsal2""",968.244446,1129.375244
9,"""D5969C1A""",13917756,"""2026-05-02T04:22:32.585646""",11020,91,76,"""ind8""","""Dorsal2""",492.474396,1011.569824
4,"""6D68A819""",13985911,"""2026-05-02T04:33:54.137450""",31466,435,269,"""ind8""","""Dorsal2""",246.589478,270.268402
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind7""","""Dorsal2""",685.252319,483.831543
…,…,…,…,…,…,…,…,…,…,…
7,"""0265E61A""",13896093,"""2026-05-02T04:18:55.935333""",4521,57,44,"""ind2""","""Dorsal2""",972.564331,778.14447
4,"""6D68A819""",13985911,"""2026-05-02T04:33:54.137450""",31466,435,269,"""ind2""","""Dorsal2""",423.872498,133.624481
5,"""F02AE61A""",13883423,"""2026-05-02T04:16:49.228317""",721,0,null,"""ind4""","""Dorsal2""",711.786011,360.191193


In [ ]:
antenna_with_coords.group_by(
    ['run_id', 'tag', 'individuals']
).agg(
    pl.col("x").mean(),
    pl.col("y").mean(),
)